In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Paths are relative to the notebooks directory
project_root = Path("..")
data_path = project_root / "reports" / "leetcode_features.csv"
reports_dir = project_root / "reports"

if not data_path.exists():
    raise FileNotFoundError(
        f"Could not find {data_path}. Run the exploration notebook "
        "and save reports/leetcode_features.csv first."
    )

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (2314, 11)


,leetcode_problem_id,python_code,code_length,syntax_valid,num_functions,num_classes,num_loops,num_conditionals,num_function_calls,num_imports,num_lines
0,01-matrix,import heapq\nfrom collections import defaultd...,1304,1,1,1,5,6,16,2,35
1,1-bit-and-2-bit-characters,"class Solution:\n def isValid(self, bits, i...",716,1,2,1,0,8,5,0,25
2,132-pattern,"class Solution:\n def find132pattern(self, ...",390,1,1,1,2,1,6,0,12
3,2-keys-keyboard,from functools import lru_cache\n\nclass Solut...,410,1,1,1,0,3,6,1,14
4,24-game,"class Solution:\n def judgePoint24(self, ca...",1122,1,2,1,7,3,22,0,27


In [2]:
candidate_features = [
    "code_length",
    "num_functions",
    "num_classes",
    "num_loops",
    "num_conditionals",
    "num_function_calls",
    "num_imports",
    "num_lines",
    "max_line_length",
    "avg_line_length",
]

feature_columns = [
    column for column in candidate_features
    if column in df.columns
]

if not feature_columns:
    raise ValueError("No expected numerical feature columns were found.")

X = df[feature_columns].apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)

print("Features used:", feature_columns)
print("Rows:", len(X))
print("Missing feature values:", int(X.isna().sum().sum()))
display(X.describe().round(2))

Features used: ['code_length', 'num_functions', 'num_classes', 'num_loops', 'num_conditionals', 'num_function_calls', 'num_imports', 'num_lines']
Rows: 2314
Missing feature values: 0


,code_length,num_functions,num_classes,num_loops,num_conditionals,num_function_calls,num_imports,num_lines
count,2314.00,2314.00,2314.00,2314.00,2314.00,2314.00,2314.00,2314.00
mean,634.94,1.55,1.04,2.17,2.05,7.18,0.26,19.58
std,739.36,1.22,0.24,1.89,2.05,5.95,0.51,13.48
min,82.00,0.00,0.00,0.00,0.00,0.00,0.00,3.00
25%,348.00,1.00,1.00,1.00,1.00,3.00,0.00,11.00
50%,511.00,1.00,1.00,2.00,2.00,6.00,0.00,16.00
75%,777.00,2.00,1.00,3.00,3.00,9.00,0.00,24.00
max,29620.00,17.00,3.00,17.00,16.00,75.00,3.00,203.00


In [3]:
model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "isolation_forest",
            IsolationForest(
                n_estimators=200,
                contamination="auto",
                random_state=42
            )
        ),
    ]
)

model.fit(X)

print("Isolation Forest training completed.")

Isolation Forest training completed.


In [4]:
predictions = model.predict(X)
anomaly_scores = model.decision_function(X)

results = df.copy()
results["anomaly_label"] = predictions
results["anomaly_score"] = anomaly_scores
results["is_anomaly"] = results["anomaly_label"] == -1

print("Not flagged:", int((predictions == 1).sum()))
print("Flagged for review:", int((predictions == -1).sum()))

display(
    results.loc[
        results["is_anomaly"],
        ["leetcode_problem_id", *feature_columns, "anomaly_score"]
    ].sort_values("anomaly_score").head(10)
)

Not flagged: 2015
Flagged for review: 299


,leetcode_problem_id,code_length,num_functions,num_classes,num_loops,num_conditionals,num_function_calls,num_imports,num_lines,anomaly_score
891,k-divisible-elements-subarrays,5955,17,1,17,15,75,0,203,-0.306144
996,longest-duplicate-substring,5639,17,1,14,15,71,0,192,-0.303965
1471,minimum-number-of-valid-strings-to-form-target-i,3439,10,3,8,5,30,1,102,-0.271743
1094,maximize-count-of-distinct-primes-after-split,3354,8,2,9,12,34,0,108,-0.266511
1664,number-of-subarrays-with-and-value-of-k,1797,14,3,3,4,32,1,67,-0.261606
1666,number-of-subarrays-with-gcd-equal-to-k,1728,15,3,4,3,31,1,69,-0.256108
1085,max-value-of-equation,1693,13,3,3,3,31,1,64,-0.255341
994,longest-continuous-subarray-with-absolute-diff...,1734,11,3,3,4,36,0,60,-0.242938
255,continuous-subarrays,1638,11,3,3,3,33,0,59,-0.239877
242,construct-string-with-minimum-cost,2367,6,3,8,8,28,0,84,-0.233959


In [5]:
# Summary of anomaly scores

print("Anomaly score statistics:")
display(results["anomaly_score"].describe().round(4))

# Compare average features between both groups

comparison = results.groupby("is_anomaly")[feature_columns].mean()

comparison.index = ["Not flagged", "Flagged"]

display(comparison.round(2))

Anomaly score statistics:


count    2314.0000
mean        0.0731
std         0.0715
min        -0.3061
25%         0.0561
50%         0.0932
75%         0.1212
max         0.1473
Name: anomaly_score, dtype: float64

,code_length,num_functions,num_classes,num_loops,num_conditionals,num_function_calls,num_imports,num_lines
Not flagged,513.64,1.29,1.00,1.83,1.69,5.87,0.19,16.14
Flagged,1452.38,3.26,1.34,4.44,4.50,15.96,0.73,42.79


In [6]:
reports_dir.mkdir(parents=True, exist_ok=True)

model_path = reports_dir / "isolation_forest_model.joblib"
results_path = reports_dir / "anomaly_detection_results.csv"

joblib.dump(
    {"model": model, "feature_columns": feature_columns},
    model_path
)
results.to_csv(results_path, index=False)

print("Model saved to:", model_path)
print("Results saved to:", results_path)

Model saved to: ../reports/isolation_forest_model.joblib
Results saved to: ../reports/anomaly_detection_results.csv
